# BD-06: validation comparison, coded judges and calibration

**Outcome:** the historical arraignment decision to set bail or remand. This is not actual detention, future offending, or a recommended decision.

This notebook compares the three published main feature sets for five models and one charge rule, using saved development predictions. Gender supplements are excluded from the main comparison. It does not fit/tune/recalibrate models, select a fairness threshold, execute the final-test path or read test-year source data. The filename is reserved for later BD-07, but every calculation in this version is validation-only.

Inputs: local `predictions_baseline.parquet`, final PR #7 `predictions_ensemble.parquet` and its companion manifest. The latter two are Release attachments at https://github.com/POL408-Team5/ny-pretrial-detention/releases/tag/e02-pr7-early-stopping and are never committed here. Put them in `data/processed/`. The final Release supersedes the fixed-400-tree exploratory output.

In [1]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append("..")
from src import config
from pathlib import Path
from importlib.metadata import version
import hashlib
import json
import gc
import numpy as np
import pandas as pd

from src.evaluate import (
    audit_development_predictions, load_predictions, metrics,
    validation_comparison, plot_validation_comparison, write_validation_report,
)
from src.sample import build_sample, load_raw

assert config.ROOT == Path.cwd().resolve().parent, "Run from the notebooks directory."
versions = {name: version(name) for name in ["pandas", "numpy", "scikit-learn", "pyarrow", "matplotlib"]}
print("Python:", sys.version.split()[0], "Packages:", versions)
for path in [config.PRED_BASELINE_FILE, config.PRED_ENSEMBLE_FILE]:
    assert path.is_file(), f"Missing local prediction artifact: {path.name}"
manifest_path = config.PRED_ENSEMBLE_FILE.with_suffix(".pr7_manifest.json")
manifest_bytes = manifest_path.read_bytes()
assert hashlib.sha256(manifest_bytes).hexdigest() == "91f2f128e9f60377e653e129e2ed4b742b74c920455800614990f0dd71af3fab"
manifest = json.loads(manifest_bytes)
assert manifest["source_commit"] == "51dd03b39aaf6fd5f76aa3d505b809f69f398232"
assert not manifest["includes_2024_test"]
def file_sha256(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()
ensemble_hash = file_sha256(config.PRED_ENSEMBLE_FILE)
assert ensemble_hash == manifest["sha256"] == "a8bddd0e548fd814954a47aa139e04bcd90fce5a039b12a054c59233a53ba908"
assert config.PRED_ENSEMBLE_FILE.stat().st_size == manifest["size_bytes"]
provenance = {
    "analysis_date": "2026-11-15", "main_commit": "81c4231df95a91e0c8440fd53171f4e98e2eb96c",
    "ensemble_source_commit": manifest["source_commit"], "ensemble_source_pr": manifest["source_pr"],
    "ensemble_release": "https://github.com/POL408-Team5/ny-pretrial-detention/releases/tag/e02-pr7-early-stopping",
    "baseline_sha256": file_sha256(config.PRED_BASELINE_FILE), "ensemble_sha256": ensemble_hash,
    "ensemble_manifest_sha256": hashlib.sha256(manifest_bytes).hexdigest(),
    "python": sys.version.split()[0], "packages": versions,
    "bootstrap_draws": config.N_BOOTSTRAP, "bootstrap_seed": config.RANDOM_STATE,
    "early_stopping": manifest["early_stopping"],
    "model_specs": json.loads((config.REPORTS / "bd06_model_specs.json").read_text(encoding="utf-8")),
}
print("Verified final early-stopping Release and local baseline hashes.")

Python: 3.11.18 Packages: {'pandas': '3.1.1', 'numpy': '2.4.6', 'scikit-learn': '1.9.1', 'pyarrow': '26.0.1', 'matplotlib': '3.11.2'}
Verified final early-stopping Release and local baseline hashes.


## Development metadata and input alignment

Read only configured development years. If the team's analytic file is absent, rebuild those years using the unchanged team sample functions and compare every development-year sample-flow step. Only case/split/label and training-year anonymous judge metadata are retained. `judge_id_final` is not used. No sample-flow report or canonical analytic file is overwritten.

In [2]:
development_years = [*config.TRAIN_YEARS, config.VALID_YEAR]
assert config.TEST_YEAR not in development_years
judge_col = config.FEATURES_JUDGE[0]
metadata_columns = [config.ID_COL, config.YEAR_COL, config.SPLIT_COL,
                    config.MAIN_SAMPLE_COL, config.TARGET_Q1, judge_col]
published_flow = pd.read_csv(config.SAMPLE_FLOW_FILE).set_index("step")
if config.ANALYTIC_FILE.is_file():
    development = pd.read_parquet(config.ANALYTIC_FILE, columns=metadata_columns,
        filters=[(config.YEAR_COL, "in", development_years)])
    provenance["metadata_source"] = "analytic.parquet: development partitions only"
else:
    raw = load_raw(years=development_years)
    assert set(raw[config.YEAR_COL].dropna().unique()) == set(development_years)
    development, flow = build_sample(raw)
    del raw
    pd.testing.assert_frame_equal(
        flow.set_index("step")[[str(y) for y in development_years]],
        published_flow[[str(y) for y in development_years]], check_dtype=False)
    development = development[metadata_columns]
    provenance["metadata_source"] = "development years rebuilt with unchanged team sample code; every flow step matched"
development = development.loc[development[config.MAIN_SAMPLE_COL]].copy()
assert set(development[config.YEAR_COL].unique()) == set(development_years)
assert config.JUDGE_COL_FINAL not in development
counts = development.groupby(config.YEAR_COL).size().to_dict()
assert counts == {year: int(published_flow.loc["MAIN", str(year)]) for year in development_years}
provenance["development_counts"] = {str(k): int(v) for k, v in counts.items()}
development.to_parquet(config.DATA_PROCESSED / "bd06_development_metadata.parquet", index=False)
gc.collect()
print("Development metadata counts:", counts)
pred_df = load_predictions()
coverage = audit_development_predictions(pred_df, development)
print("Prediction alignment verified:")
print(coverage.to_string(index=False))
provenance["prediction_rows_supplied"] = len(pred_df)
provenance["all_case_ids_labels_and_years_matched"] = True
print("No case records or judge identities are displayed.")

  read NYS for Web 2021.csv.gz: 246,202 rows (124,264 NYC)
  read NYS for Web 2022.csv.gz: 286,588 rows (150,585 NYC)
  read NYS for Web 2023.csv.gz: 324,619 rows (175,658 NYC)
Development metadata counts: {np.int64(2021): 89189, np.int64(2022): 104007, np.int64(2023): 120457}
Prediction alignment verified:
model      feature_set     split      n
 rule             CASE train_oof 193196
 rule             CASE     valid 120457
logit             CASE train_oof 193196
logit             CASE     valid 120457
logit       CASE_COURT train_oof 193196
logit       CASE_COURT     valid 120457
logit CASE_COURT_JUDGE train_oof 193196
logit CASE_COURT_JUDGE     valid 120457
logit CASE_PLUS_GENDER train_oof 193196
logit CASE_PLUS_GENDER     valid 120457
lasso             CASE train_oof 193196
lasso             CASE     valid 120457
lasso       CASE_COURT train_oof 193196
lasso       CASE_COURT     valid 120457
lasso CASE_COURT_JUDGE train_oof 193196
lasso CASE_COURT_JUDGE     valid 120457
lasso CASE_

In [3]:
# Recompute the published input metrics before any bootstrap analysis.
input_metrics = pd.DataFrame([
    {"model": model, "feature_set": fs, "split": split, **metrics(g.y_true, g.y_prob)}
    for (model, fs, split), g in pred_df.groupby(["model", "feature_set", "split"], sort=False)
]).set_index(["model", "feature_set", "split"])
ensemble_max_error = 0.0
for expected in manifest["metrics"]:
    got = input_metrics.loc[(expected["model"], expected["feature_set"], expected["split"])]
    assert int(got["n"]) == expected["n"]
    for metric in ["roc_auc", "avg_precision", "brier", "base_rate"]:
        error = abs(float(got[metric]) - expected[metric])
        ensemble_max_error = max(ensemble_max_error, error)
        assert error <= 1e-12
baseline_reference = pd.read_csv(config.REPORTS / "bd05_metrics.csv")
baseline_reference = baseline_reference.loc[
    baseline_reference.split.eq("valid") & baseline_reference.feature_set.isin(config.FEATURE_SETS)]
baseline_max_error = 0.0
for expected in baseline_reference.to_dict("records"):
    got = input_metrics.loc[(expected["model"], expected["feature_set"], "valid")]
    assert int(got["n"]) == expected["n"]
    for metric in ["roc_auc", "avg_precision", "brier", "base_rate"]:
        error = abs(float(got[metric]) - expected[metric])
        baseline_max_error = max(baseline_max_error, error)
        assert error <= 1e-9, "Baseline predictions differ materially from the committed main validation report."
provenance["ensemble_metric_rows_recomputed"] = len(manifest["metrics"])
provenance["ensemble_metric_max_error"] = ensemble_max_error
provenance["baseline_main_validation_metric_max_error"] = baseline_max_error
provenance["baseline_note"] = "Original BD-05 local predictions retained. Main validation scores reproduce; supplementary gender and version-sensitive OOF values do not select the main model."
print("All final ensemble scores and main baseline validation scores reproduced.")
print("Maximum differences:", {"ensemble": ensemble_max_error, "baseline": baseline_max_error})

All final ensemble scores and main baseline validation scores reproduced.
Maximum differences: {'ensemble': 1.1102230246251565e-16, 'baseline': 1.1797784971179226e-12}


## Paired validation comparison

All models are evaluated on the same validation cases, then on the same individually coded-judge subset (`judge_id` starts with J). Five main model families and the charge rule are retained; gender supplements are not ranked. The configured number of ordinary docket bootstrap draws is shared across every prediction column within a cohort. Multiplicity-weighted AUC is exactly equivalent to rescoring the sampled repeated rows, including tied scores, and is tested against sklearn and the existing paired-bootstrap helper.

Intervals condition on fitted models, are not judge-cluster or model-selection-adjusted intervals, and do not establish causal judge effects. No model is retrained or recalibrated here.

In [4]:
results = validation_comparison(pred_df, development, progress=True)
print("Cohorts:")
print(results["cohorts"].to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print("Validation comparison, including marginal AUC intervals:")
print(results["metrics"].to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print("Paired nested feature gains (larger minus smaller):")
print(results["increments"].to_string(index=False, float_format=lambda x: f"{x:+.6f}"))
print("Candidate XGBoost versus logistic baselines:")
print(results["model_differences"].to_string(index=False, float_format=lambda x: f"{x:+.6f}"))

Cohort all_valid: n=120,457
  paired bootstrap: 200/1000
  paired bootstrap: 400/1000
  paired bootstrap: 600/1000
  paired bootstrap: 800/1000
  paired bootstrap: 1000/1000
Cohort coded_judges: n=76,680
  paired bootstrap: 200/1000
  paired bootstrap: 400/1000
  paired bootstrap: 600/1000
  paired bootstrap: 800/1000
  paired bootstrap: 1000/1000
Cohorts:
      cohort      n  n_positive  base_rate  share_of_valid
   all_valid 120457       15440   0.128179        1.000000
coded_judges  76680       10793   0.140754        0.636576
Validation comparison, including marginal AUC intervals:
      cohort model      feature_set  roc_auc  avg_precision    brier      n  base_rate  mean_pred  calibration_gap  auc_ci_low  auc_ci_high  bootstrap_draws
   all_valid  rule             CASE 0.846492       0.478021 0.080927 120457   0.128179   0.127613        -0.000566    0.843344     0.849698             1000
   all_valid logit             CASE 0.920289       0.662465 0.067290 120457   0.128179   0.11

In [5]:
plot_validation_comparison(results)
report_path = write_validation_report(results, provenance)
print("Aggregate report:", report_path.relative_to(config.ROOT))
print("Figures: validation AUC, paired nested gains, calibration (PNG and SVG).")
print("Mean prediction minus actual prevalence:")
print(results["metrics"][["cohort", "model", "feature_set", "mean_pred", "base_rate", "calibration_gap", "brier"]]
    .to_string(index=False, float_format=lambda x: f"{x:.6f}"))
assert set(pred_df["split"]) == {"train_oof", "valid"}
assert not pred_df[config.YEAR_COL].eq(config.TEST_YEAR).any()
print("BD-06 comparison and recommendation complete; group model/threshold freeze remains pending.")
print("No model fitting, no threshold selection, no 2024 raw file or test prediction used.")

Aggregate report: reports\bd06_validation_report.md
Figures: validation AUC, paired nested gains, calibration (PNG and SVG).
Mean prediction minus actual prevalence:
      cohort model      feature_set  mean_pred  base_rate  calibration_gap    brier
   all_valid  rule             CASE   0.127613   0.128179        -0.000566 0.080927
   all_valid logit             CASE   0.119961   0.128179        -0.008217 0.067290
   all_valid logit       CASE_COURT   0.119655   0.128179        -0.008523 0.067026
   all_valid logit CASE_COURT_JUDGE   0.120279   0.128179        -0.007899 0.065788
   all_valid lasso             CASE   0.119984   0.128179        -0.008194 0.067270
   all_valid lasso       CASE_COURT   0.119641   0.128179        -0.008538 0.067029
   all_valid lasso CASE_COURT_JUDGE   0.121306   0.128179        -0.006873 0.065698
   all_valid  tree             CASE   0.120376   0.128179        -0.007802 0.067615
   all_valid  tree       CASE_COURT   0.119983   0.128179        -0.008195 0.0

## Handoff

Use the all-case validation table as the primary result; the individually coded-judge subset diagnoses coverage and cannot replace it. Report both positive and negative increments. C supplies thresholds and protected-group errors; E supplies feature explanations. The group must confirm the candidate, hyperparameters, preprocessing and stopping/refit policy before BD-07. `run_test=True` is still blocked. This notebook does not silently implement or execute final testing.

The complete interpretation, uncertainty limitations and freeze checklist are in `reports/bd06_validation_report.md`. Do not commit raw data, case predictions, private metadata or bootstrap draws.